In [0]:
use catalog pricing_analytics;

create or replace table silver.reporting_dim_state_stage_1 as(
  select
  distinct STATE_NAME
  from silver.daily_pricing_silver
  WHERE lakehouse_update_date > (SELECT nvl(MAX(PROCESSED_TABLE_DATETIME),'1900-01-01') FROM processrunlogs.DELTALAKEHOUSE_PROCESS_RUNS
WHERE PROCESS_NAME = 'reportingDimensionTablesLoad' AND PROCESS_STATUS = 'completed')
)

num_affected_rows,num_inserted_rows


In [0]:
create or replace table silver.reporting_dim_state_stage_2 as(
select
silverDim.STATE_NAME,
row_number() over(order by silverDim.STATE_NAME) as STATE_ID,
current_timestamp() as lakehouse_inserted_date,
current_timestamp() as lakehouse_updated_date
from silver.reporting_dim_state_stage_1 silverDim
left outer join gold.reporting_dim_state_gold goldDim
on silverDim.STATE_NAME = goldDim.STATE_NAME
where goldDim.STATE_NAME is null)

num_affected_rows,num_inserted_rows


In [0]:
create or replace table silver.reporting_dim_state_stage_3 as (
  select
  silverDim.STATE_NAME, 
  silverDim.STATE_ID + PREV_MAX_SK_ID as STATE_ID,
  current_timestamp() as lakehouse_inserted_date,
  current_timestamp() as lakehouse_updated_date
  from silver.reporting_dim_state_stage_2 silverDim
  cross join(select nvl(max(STATE_ID),0) as PREV_MAX_SK_ID from gold.reporting_dim_state_gold)) 

num_affected_rows,num_inserted_rows


In [0]:
insert into gold.reporting_dim_state_gold
select
STATE_NAME,
STATE_ID,
current_timestamp(),
current_timestamp()
from silver.reporting_dim_state_stage_3

num_affected_rows,num_inserted_rows
0,0


In [0]:
USE CATALOG pricing_analytics;
CREATE OR REPLACE TABLE silver.reporting_dim_market_stage_1 AS
SELECT 
 DISTINCT MARKET_NAME
FROM silver.daily_pricing_silver
WHERE lakehouse_update_date > (SELECT nvl(max(PROCESSED_TABLE_DATETIME),'1900-01-01') FROM processrunlogs.DELTALAKEHOUSE_PROCESS_RUNS 
WHERE process_name = 'reportingDimensionTablesLoad' AND process_status = 'Completed' )

num_affected_rows,num_inserted_rows


In [0]:
CREATE OR REPLACE TABLE silver.reporting_dim_market_stage_2 AS 
SELECT 
  silverDim.MARKET_NAME
 ,ROW_NUMBER() OVER (  ORDER BY silverDim.MARKET_NAME)  as MARKET_ID
 ,current_timestamp() as lakehouse_inserted_date
 ,current_timestamp() as lakehouse_updated_date
FROM silver.reporting_dim_market_stage_1 silverDim
LEFT OUTER JOIN gold.reporting_dim_market_gold goldDim
ON silverDim.MARKET_NAME = goldDim.MARKET_NAME
WHERE goldDim.MARKET_NAME IS NULL;


num_affected_rows,num_inserted_rows


In [0]:
CREATE OR REPLACE TABLE silver.reporting_dim_market_stage_3 AS 
SELECT
silverDim.MARKET_NAME 
,silverDim.MARKET_ID + PREV_MAX_SK_ID as MARKET_ID
,current_timestamp() as lakehouse_inserted_date
,current_timestamp() as lakehouse_updated_date
FROM 
silver.reporting_dim_market_stage_2 silverDim
CROSS JOIN (SELECT NVL(MAX(MARKET_ID),0) as PREV_MAX_SK_ID FROM gold.reporting_dim_market_gold ) goldDim;

num_affected_rows,num_inserted_rows


In [0]:
INSERT INTO gold.reporting_dim_market_gold
SELECT
MARKET_NAME
,MARKET_ID
,current_timestamp() 
,current_timestamp() 
FROM silver.reporting_dim_market_stage_3;

num_affected_rows,num_inserted_rows
1664,1664


In [0]:
CREATE OR REPLACE TABLE silver.reporting_dim_variety_stage_1 AS
SELECT 
 DISTINCT VARIETY
FROM silver.daily_pricing_silver
WHERE lakehouse_update_date > (SELECT nvl(max(PROCESSED_TABLE_DATETIME),'1900-01-01') FROM processrunlogs.DELTALAKEHOUSE_PROCESS_RUNS 
WHERE process_name = 'reportingDimensionTablesLoad' AND process_status = 'Completed' );


num_affected_rows,num_inserted_rows


In [0]:
CREATE OR REPLACE TABLE silver.reporting_dim_variety_stage_2 AS 
SELECT 
  silverDim.VARIETY
 ,ROW_NUMBER() OVER (  ORDER BY silverDim.VARIETY)  as VARIETY_ID
 ,current_timestamp() as lakehouse_inserted_date
 ,current_timestamp() as lakehouse_updated_date
FROM silver.reporting_dim_variety_stage_1 silverDim
LEFT OUTER JOIN gold.reporting_dim_variety_gold goldDim
ON silverDim.VARIETY= goldDim.VARIETY
WHERE goldDim.VARIETY IS NULL;

num_affected_rows,num_inserted_rows


In [0]:
CREATE OR REPLACE TABLE silver.reporting_dim_variety_stage_3 AS 
SELECT
silverDim.VARIETY 
,silverDim.VARIETY_ID + PREV_MAX_SK_ID as VARIETY_ID
,PREV_MAX_SK_ID
,current_timestamp() as lakehouse_inserted_date
,current_timestamp() as lakehouse_updated_date
FROM 
silver.reporting_dim_variety_stage_2 silverDim
CROSS JOIN (SELECT nvl(MAX(VARIETY_ID),0) as PREV_MAX_SK_ID FROM gold.reporting_dim_variety_gold ) goldDim;



num_affected_rows,num_inserted_rows


In [0]:
INSERT INTO gold.reporting_dim_variety_gold
SELECT
VARIETY
,VARIETY_ID
,current_timestamp() 
,current_timestamp() 
FROM silver.reporting_dim_variety_stage_3

num_affected_rows,num_inserted_rows
417,417


In [0]:
CREATE OR REPLACE TABLE silver.reporting_dim_product_stage_1 AS
SELECT 
 DISTINCT PRODUCT_NAME
 ,PRODUCTGROUP_NAME
FROM silver.daily_pricing_silver
WHERE lakehouse_update_date > (SELECT nvl(max(PROCESSED_TABLE_DATETIME),'1900-01-01') FROM processrunlogs.DELTALAKEHOUSE_PROCESS_RUNS 
WHERE process_name = 'reportingDimensionTablesLoad' AND process_status = 'Completed' );

num_affected_rows,num_inserted_rows


In [0]:
CREATE OR REPLACE TABLE silver.reporting_dim_product_stage_2 AS 
SELECT 
  silverDim.PRODUCT_NAME
  ,silverDim.PRODUCTGROUP_NAME
 ,ROW_NUMBER() OVER (  ORDER BY silverDim.PRODUCT_NAME,silverDim.PRODUCTGROUP_NAME)  as PRODUCT_ID
 ,current_timestamp() as lakehouse_inserted_date
 ,current_timestamp() as lakehouse_updated_date
FROM silver.reporting_dim_product_stage_1 silverDim
LEFT OUTER JOIN gold.reporting_dim_product_gold goldDim
ON silverDim.PRODUCT_NAME= goldDim.PRODUCT_NAME
AND silverDim.PRODUCTGROUP_NAME = goldDim.PRODUCTGROUP_NAME
WHERE goldDim.PRODUCT_NAME IS NULL;

num_affected_rows,num_inserted_rows


In [0]:
CREATE OR REPLACE TABLE silver.reporting_dim_product_stage_3 AS 
SELECT
  silverDim.PRODUCT_NAME
  ,silverDim.PRODUCTGROUP_NAME
,silverDim.PRODUCT_ID + PREV_MAX_SK_ID as PRODUCT_ID
,PREV_MAX_SK_ID
,current_timestamp() as lakehouse_inserted_date
,current_timestamp() as lakehouse_updated_date
FROM 
silver.reporting_dim_product_stage_2 silverDim
CROSS JOIN (SELECT nvl(MAX(PRODUCT_ID),0) as PREV_MAX_SK_ID FROM gold.reporting_dim_product_gold ) goldDim;

num_affected_rows,num_inserted_rows


In [0]:
INSERT INTO gold.reporting_dim_product_gold
SELECT
 PRODUCTGROUP_NAME
,PRODUCT_NAME
,PRODUCT_ID
,current_timestamp() 
,current_timestamp() 
FROM silver.reporting_dim_product_stage_3

num_affected_rows,num_inserted_rows
215,215


In [0]:
INSERT INTO  processrunlogs.DELTALAKEHOUSE_PROCESS_RUNS(PROCESS_NAME,PROCESSED_TABLE_DATETIME,PROCESS_STATUS)
SELECT 'reportingDimensionTablesLoad' , max(lakehouse_update_date) ,'Completed' FROM silver.daily_pricing_silver

num_affected_rows,num_inserted_rows
1,1
